# Kinthread RL on a free Colab T4
Reinforcement learning with verifiable rewards (`train_rlvr.py`) on keyed questions the reader never trained on: symgen (Gemini-written, keyed by agreement) and new crawled questions (`real_train2_pool.jsonl` in the shared folder, if present). Every draw is a freshly altered copy; selection and early stop by real dev questions only. Run all; it resumes nothing (short run, best.pt kept on Drive).

In [ ]:
import os, torch
assert torch.cuda.is_available(), 'No GPU: Runtime > Change runtime type > T4 GPU'
REPO = '/content/Kinthread-llm-model'
if not os.path.exists(REPO):
    !git clone --depth 1 -b v9-reader https://github.com/ArAnirudh2901/Kinthread-llm-model.git {REPO}
else:
    !git -C {REPO} pull --ff-only
%cd {REPO}
from google.colab import drive
drive.mount('/content/drive')
BASE = next(b for b in ('/content/drive/MyDrive/kinthread_shared', '/content/drive/MyDrive/kinthread') if os.path.isdir(b))
INIT_RUN = 'v10-r6a' if os.path.exists(f'{BASE}/runs/v10-r6a/best.pt') else 'v10-r5-plain'
RUN = f'{INIT_RUN}-rl'
print('init', INIT_RUN, '->', RUN)
os.makedirs('data/real_questions', exist_ok=True)
!tar -xzf {BASE}/real_questions.tgz -C data/real_questions
POOL = 'data/symgen/*.jsonl.gz' + (f' {BASE}/real_train2_pool.jsonl' if os.path.exists(f'{BASE}/real_train2_pool.jsonl') else '')

In [ ]:
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python train_rlvr.py --init {BASE}/runs/{INIT_RUN}/best.pt --pool {POOL} --out {BASE}/runs/{RUN} --steps 1500 --questions 16 --versions 2 --group 4 --real_dev 300 --eval_every 100 --patience 5 --amp

In [ ]:
import os
os.makedirs('data/real_questions', exist_ok=True)
!python -m eval.run_real --ckpt {BASE}/runs/{RUN}/best.pt --split dev --out {BASE}/runs/{RUN}/real_dev.json | tail -4